In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import math

# Muat data
ratings = pd.read_csv('../data/processed/ratings_clean.csv') # pastikan sudah bersih
# Buat user-item matrix (user = baris, film = kolom, nilai = rating)
user_item_matrix = ratings.pivot(index='userId', columns='movieId', values='rating').fillna(0)

# Hitung similaritas antar item (film)
item_similarity = cosine_similarity(user_item_matrix.T)  # transpose agar item jadi baris
item_similarity_df = pd.DataFrame(item_similarity, index=user_item_matrix.columns, columns=user_item_matrix.columns)

# Fungsi prediksi rating user u untuk film i (weighted sum)
def predict_rating(user_id, movie_id, k=20):
    if movie_id not in item_similarity_df.columns:
        return np.nan
    user_ratings = user_item_matrix.loc[user_id]
    sim_scores = item_similarity_df[movie_id]
    rated_movies = user_ratings[user_ratings > 0].index
    sim_scores = sim_scores[rated_movies]
    if len(sim_scores) == 0:
        return np.nan
    top_k = sim_scores.sort_values(ascending=False).head(k)
    pred = np.dot(top_k, user_ratings[top_k.index]) / top_k.sum()
    return pred

# Evaluasi dengan split data
train, test = train_test_split(ratings, test_size=0.2, random_state=42)
# (Latih model tidak perlu training, langsung evaluasi di test set)
test['predicted'] = test.apply(lambda row: predict_rating(row['userId'], row['movieId']), axis=1)
test = test.dropna(subset=['predicted'])
rmse = math.sqrt(mean_squared_error(test['rating'], test['predicted']))
print(f'Baseline Item-Based CF RMSE: {rmse:.4f}')

# 🤖 Model Baseline: Item-Based Collaborative Filtering (Memory-Based)

Pada tahap ini, kita membangun model *baseline* (model acuan dasar) menggunakan pendekatan Item-Based Collaborative Filtering. Ide utama dari pendekatan ini adalah: *"Jika seorang pengguna menyukai Film A, dan Film A memiliki kemiripan karakteristik penilaian dengan Film B, maka kemungkinan besar pengguna tersebut juga akan menyukai Film B."*

---

## ⚙️ Alur Kerja Model (Pipeline)

### 1. Pembentukan Matriks User-Item (*User-Item Matrix*)
Data interaksi rating diubah ke dalam bentuk matriks 2 dimensi menggunakan `pd.pivot()`, di mana:
* Baris (Index): `userId` (merepresentasikan setiap pengguna).
* Kolom (Columns): `movieId` (merepresentasikan setiap film).
* Nilai (Values): Rating yang diberikan pengguna. Nilai yang kosong (*missing values* / film yang belum ditonton) diisi dengan angka `0`.

### 2. Perhitungan Similaritas Kosinus (*Cosine Similarity*)
Kita menghitung tingkat kemiripan antar film berdasarkan pola rating yang diberikan oleh seluruh pengguna. Karena kita ingin menghitung kemiripan antar item (film), matriks di-*transpose* (`.T`) terlebih dahulu agar film berada di posisi baris. Formula *Cosine Similarity* yang digunakan adalah:

### 3. Prediksi Rating dengan Weighted Sum (Top-K Neighbors)
Untuk memprediksi berapa rating yang akan diberikan oleh User  terhadap Film, kita menggunakan metode rata-rata berbobot (*weighted sum*) dari Top-K film paling mirip (k=20) yang sudah dirating oleh *User* tersebut. Formulasi matematiknya:

4. Evaluasi Model (RMSE)
Model dievaluasi menggunakan metrik RMSE (Root Mean Squared Error) untuk mengukur seberapa besar penyimpangan antara nilai prediksi model dengan nilai rating asli pada *test set*.

Semakin kecil nilai RMSE (semakin mendekati 0), maka akurasi prediksi rating model semakin baik dan dapat dijadikan tolok ukur (*benchmark*) untuk model *Deep Learning* selanjutnya.

---

## Kelebihan & Keterbatasan Model Baseline Ini
* Kelebihan: Sangat intuitif, mudah dijelaskan (explainable), dan tidak memerlukan waktu training/training bobot seperti *Neural Network* (lazy learning).
* Keterbatasan:
  * Sparsity & Scalability: Ketika jumlah film dan user mencapai puluhan ribu, ukuran matriks akan sangat besar dan memakan banyak memori RAM.
  * Cold-Start Problem: Model akan menghasilkan nilai `NaN` jika film target belum pernah mendapat rating sama sekali, atau jika user belum pernah merating film yang memiliki kemiripan dengan film target.